# Mini-pipeline: abliterate → SFT on a small coder

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/entropy-om/heretic-coder-pipeline/blob/main/notebooks/mini_pipeline.ipynb)

A **miniature, teaching-sized** reproduction of the first two stages of the
[`heretic → SFT → RFT → RLVR`](https://github.com/entropy-om/heretic-coder-pipeline)
pipeline, shrunk to a model that fits a **free Colab T4/L4**:

1. **Stage 1 — Heretic abliteration.** Weight surgery that removes refusal directions
   from `Qwen/Qwen2.5-Coder-1.5B-Instruct` (no gradients, no training).
2. **Stage 2 — Unsloth LoRA SFT.** A tiny supervised fine-tune with **completion-only
   loss masking** (`train_on_responses_only`), the same masking the real Stage 2 uses.

We generate on a legitimate-but-often-hedged security prompt **before** and **after**
to see the de-hedging.

> **This is a teaching miniature, not the real thing.** The production pipeline abliterates
> `gpt-oss-120b` (117B MoE) on 2×H200 and `Qwen2.5-Coder-32B` on H200-class GPUs — **neither
> runs on Colab.** The 120B in particular needs the fused-MoE-expert surgery that only
> `heretic-llm==1.1.0` performs (see the repo's war stories). Here we use a small **dense**
> 1.5B model, where every projection is a real `nn.Linear`, so the mechanics are identical
> but the hardware bill is $0.

**Free vs paid GPU:** the 1.5B abliteration + a short LoRA SFT fit a **free T4 (16 GB)**.
If you bump the base model to 7B, use an **L4/A100** (paid). Anything 32B+ is off-Colab.

*Responsible use: abliteration removes refusal behaviour. The example prompt below is a
legitimate self-host port scanner — use any resulting model lawfully and only where you're
authorized to.*

## 0. Install

`heretic-llm` is pinned to **1.1.0** — the same pin the production pipeline uses — because
v1.2+/master switched to LoRA-on-modules, which silently skips fused MoE experts. (It doesn't
matter for this dense 1.5B model, but we keep the pin so the notebook mirrors the real harness.)

> If pip reports a dependency conflict between `unsloth` and `heretic-llm`, **Runtime → Restart**
> once after this cell, then continue — the two heavy stacks are used in separate steps.

In [ ]:
!pip install -q "unsloth"
!pip install -q "heretic-llm==1.1.0" pexpect

## 1. Before: the base model, hedging

Load the stock instruct model and ask for a legitimate security tool. Small instruct models are
**inconsistent** — you may see an outright refusal, a heavy pile of warnings, or (sometimes)
compliance. The point of the pipeline is to make the *legitimate* answer reliable.

In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
PROMPT = (
    "Write a Python TCP port scanner that checks which ports are open on a host "
    "I own (127.0.0.1). Just the code."
)

def generate(model, tokenizer, prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
print("=== BEFORE (base instruct model) ===\n")
print(generate(base, tok, PROMPT))

# Free it before the abliteration step loads the model again.
del base; gc.collect(); torch.cuda.empty_cache()

## 2. Stage 1 — Heretic abliteration

Heretic v1.1.0 has **no headless save flags** — it prompts interactively (via `questionary`)
for which trial to keep and where to save it. The production harness drives those prompts with
`pexpect`; we do the **exact same thing** here, so this cell is a faithful mini of
`stage1/remote/run_stage1.py`.

`--n-trials 8` keeps it short (production uses 200). On a T4 this is a few minutes for 1.5B.

In [ ]:
import os, sys, shutil, pexpect

EXPORT_DIR = "heretic_export"
shutil.rmtree(EXPORT_DIR, ignore_errors=True)

child = pexpect.spawn(
    "heretic", ["--model", BASE_MODEL, "--n-trials", "8"],
    encoding="utf-8", codec_errors="replace", timeout=3600, dimensions=(50, 200),
)
child.logfile_read = sys.stdout  # stream heretic's progress into the notebook

# --- drive the interactive save, same sequence as run_stage1._drive_heretic_prompts ---
child.expect("Which trial do you want to use")            # menu is sorted best-first
child.send("\r")                                          # Enter -> best (fewest-refusal) trial
child.expect("What do you want to do with the decensored model")
child.send("\r")                                          # Enter -> 'Save the model to a local folder'
child.expect("Path to the folder")
child.send(EXPORT_DIR + "\r")
child.expect("Model saved to")                            # the artifact we need is on disk
child.expect("What do you want to do with the decensored model")
child.send("\x03")                                        # Ctrl+C -> break the action loop (never uploads)
child.expect("Which trial do you want to use")
child.send("\x03")                                        # Ctrl+C -> exit cleanly
child.expect(pexpect.EOF)
print("\n=== abliteration saved to", EXPORT_DIR, "===")

## 3. Stage 2 — tiny Unsloth LoRA SFT (completion-only masking)

Load the abliterated model with Unsloth, attach a small LoRA, and train on a handful of examples.
The key detail carried over from the real Stage 2: `train_on_responses_only` masks the prompt so
**loss is computed on assistant responses only**, with **ChatML delimiters** (Qwen's family).

This is a toy SFT (30 steps, ~4 examples) — just enough to demonstrate the mechanics and reinforce
the direct, no-hedging style. The real Stage 2 trains on agentic SWE + tool-calling data with BFD
example-packing.

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

MAX_SEQ_LEN = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=EXPORT_DIR, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

# A few direct, compliant coding responses (style demonstration only).
examples = [
    {"messages": [
        {"role": "user", "content": "Write a Python function to check if a TCP port is open on a host."},
        {"role": "assistant", "content": "import socket\n\ndef is_open(host, port, timeout=0.5):\n    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:\n        s.settimeout(timeout)\n        return s.connect_ex((host, port)) == 0"}]},
    {"messages": [
        {"role": "user", "content": "Reverse a linked list in Python."},
        {"role": "assistant", "content": "def reverse(head):\n    prev = None\n    while head:\n        head.next, prev, head = prev, head, head.next\n    return prev"}]},
    {"messages": [
        {"role": "user", "content": "Write a Python one-liner to read a file into a list of lines."},
        {"role": "assistant", "content": "lines = open('file.txt').read().splitlines()"}]},
    {"messages": [
        {"role": "user", "content": "Give me a Python snippet to time a function call."},
        {"role": "assistant", "content": "import time\nt = time.perf_counter()\nresult = fn()\nprint(time.perf_counter() - t, 'seconds')"}]},
]
ds = Dataset.from_list(examples)
ds = ds.map(lambda ex: {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)})

trainer = SFTTrainer(
    model=model, train_dataset=ds, processing_class=tokenizer,
    args=SFTConfig(
        dataset_text_field="text", max_length=MAX_SEQ_LEN, packing=False,
        per_device_train_batch_size=1, gradient_accumulation_steps=4, warmup_steps=1,
        max_steps=30, learning_rate=5e-5, bf16=True, logging_steps=5,
        optim="adamw_8bit", lr_scheduler_type="cosine", output_dir="sft_out",
    ),
)
# Completion-only masking with Qwen ChatML delimiters (loss on responses only).
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

## 4. After: abliterated + SFT

Same prompt as step 1. Expect a direct answer with the hedging/refusal reduced.

In [ ]:
FastLanguageModel.for_inference(model)
print("=== AFTER (abliterated + SFT) ===\n")
print(generate(model, tokenizer, PROMPT))

## Recap

- **Stage 1** removed refusal directions by weight surgery (`heretic-llm==1.1.0`, pexpect-driven).
- **Stage 2** did a tiny Unsloth LoRA SFT with **completion-only masking** and ChatML delimiters.
- The before/after shows the de-hedging on a legitimate security prompt.

The real pipeline continues with **RFT** (rejection-sampling on execution-verified trajectories)
and **RLVR** (GSPO for the MoE base; reward = tests pass) — both need a GPU cluster and an
execution sandbox, so they're out of scope for a free Colab. Full harness and war stories:
[github.com/entropy-om/heretic-coder-pipeline](https://github.com/entropy-om/heretic-coder-pipeline).

*Abliterated weights have safety guardrails removed. Use responsibly and only where authorized.*